In [ ]:
import torch
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig

In [ ]:
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# SFT 
SFT_DATASET = "databricks/databricks-dolly-15k"
SFT_MAX_SAMPLES = 3000 

# RM
RM_DATASET = "Anthropic/hh-rlhf"
RM_MAX_SAMPLES = 5000

PPO_PROMPT_DATASET = "databricks/databricks-dolly-15k"
PPO_MAX_PROMPTS = 1000

# LORA
LORA_R = 16 
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training 
MAX_SEQ_LEN = 512
 
SFT_OUTPUT_DIR = "./checkpoints/sft"
SFT_EPOCHS = 1
SFT_BATCH_SIZE = 4
SFT_LR = 2e-4
 
RM_OUTPUT_DIR = "./checkpoints/reward_model"
RM_EPOCHS = 1
RM_BATCH_SIZE = 4
RM_LR = 2e-4
 
PPO_OUTPUT_DIR = "./checkpoints/ppo"
PPO_STEPS = 200                 # number of PPO update steps, not epochs
PPO_BATCH_SIZE = 8
PPO_MINI_BATCH_SIZE = 2
PPO_LR = 1.41e-5
PPO_CLIP_EPS = 0.2              # this is the epsilon from L^CLIP
PPO_KL_COEF = 0.05              # penalty for drifting too far from the SFT policy

### SFT 

In [ ]:
imp %pip install --upgrade torchao

In [ ]:
from trl import SFTConfig, SFTTrainer

def format_example(example):
    instruction = example["instruction"]
    context = example.get("context", "")
    response = example["response"]


    if context:
        prompt = f"### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"

    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    return {"text": prompt + response}

def main():
    print(f"Loading base model: {BASE_MODEL}")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    if tokenizer.pad_token is None: 
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        device_map = "auto"
    )

    dataset = load_dataset(SFT_DATASET, split = "train")
    
    if SFT_MAX_SAMPLES:
        dataset = dataset.select(range(min(SFT_MAX_SAMPLES, len(dataset)))) 

    dataset = dataset.map(format_example)

    lora_config = LoraConfig(
        r = LORA_R, 
        lora_alpha = LORA_ALPHA,
        lora_dropout = LORA_DROPOUT,
        target_modules = LORA_TARGET_MODULES, 
        bias = "none",
        task_type = "CAUSAL_LM",
    )                                             

    sft_args = SFTConfig(
        output_dir = SFT_OUTPUT_DIR,
        num_train_epochs = SFT_EPOCHS, 
        per_device_train_batch_size = SFT_BATCH_SIZE, 
        gradient_accumulation_steps = 4,
        learning_rate = SFT_LR,
        logging_steps = 10,
        save_strategy = "epoch",
        bf16 = torch.cuda.is_bf16_supported(),
        fp16 = not torch.cuda.is_bf16_supported(),
        max_length = MAX_SEQ_LEN,
        dataset_text_field = "text",
        report_to = "none",
    )

    sft_trainer = SFTTrainer(
        model = model,
        args = sft_args,
        train_dataset = dataset, 
        peft_config = lora_config, 
        processing_class = tokenizer,
    )


    sft_trainer.train()
    sft_trainer.save_model(SFT_OUTPUT_DIR)
    tokenizer.save_pretrained(SFT_OUTPUT_DIR)

    print(f"SFT model saved to {SFT_OUTPUT_DIR}")



main()


Loading base model: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.912836
20,1.632237
30,1.708595
40,1.658094
50,1.765187


In [1]:
%pip install trl

/Users/jeldy/Documents/AUTODICDACT/AI:DL:ML:RAG:Agent/research_dojo/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig 

def format_paris(example):
    return {"chosen": example["chosen"], "rejected": example["rejected"]}

rm_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if rm_tokenizer.pad_token is None:
    rm_tokenizer.pad_token = rm_tokenizer.eos_token


reward_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, 
    num_labels = 1, 
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16, 
    device_map = "auto",
)

reward_model.config.pad_token_id = rm_tokenizer.pad_token

rm_dataset = load_dataset(RM_DATASET, split = "train")

if RM_MAX_SAMPLES: 
    rm_dataset = rm_dataset.select(range(min(RM_MAX_SAMPLES, len(rm_dataset))))

rm_dataset = rm_dataset.map(format_paris)

rm_lora_config = LoraConfig(
    r = LORA_R,
    lora_alpha = LORA_ALPHA, 
    lora_dropout = LORA_DROPOUT, 
    target_modules = LORA_TARGET_MODULES, 
    bias = "none",
    task_type = "SEQ_CLS"
)

rm_args = RewardConfig(
    output_dir = RM_OUTPUT_DIR, 
    num_train_epochs = RM_EPOCHS,
    per_device_train_batch_size = RM_BATCH_SIZE,
    gradient_accumulation_steps = 4, 
    learning_rates = RM_LR,
    logging_steps = 10, 
    save_strategy = "epoch",
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),
    max_lenght = MAX_SEQ_LEN,
    report_to = "none",
)


rm_trainer = RewardTrainer(
    model = reward_model, 
    args = rm_args, 
    train_dataset = rm_dataset, 
    peft_config = rm_lora_config, 
    processing_class = rm_tokenizer,
)

rm_trainer.train()
rm_tokenizer.save_model(RM_OUTPUT_DIR)
rm_tokenizer.save_pretrained(RM_OUTPUT_DIR)
print(f"Reward model saved to {RM_OUTPUT_DIR}")


In [ ]:
from trl import AutoModelForCausalLMWithValueHead, PPOTrainer, PPOConfig 

ppo_tokenizer = AutoTokenizer.from_pretrained(SFT_OUTPUT_DIR)

if ppo_tokenizer.pad_token is None: 
    ppo_tokenizer = ppo_tokenizer.eos_token

policy_model = AutoModelForCausalLMWithValueHead.from_pretrained( 
    SFT_OUTPUT_DIR,
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    SFT_OUTPUT_DIR,
    dtype = torch.bfloat16 if torch.cuda.is_supported() else torch.float16,
)

reward_model_ppo = AutoModelForSequenceClassification.from_pretrained(
    RM_OUTPUT_DIR, 
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported else torch.float16
)

reward_model_ppo.eval()

def build_prompt_dataset(tokenizer):
    ds = load_dataset(PPO_PROMPT_DATASET, split = "train")
    
    if PPO_MAX_PROMPTS:
        ds = ds.select(range(min(PPO_MAX_PROMPTS, len(ds))))

    def tok(example):
        prompt = f"### Instruction:\n{example["instruction"]}\n\n### Response:\n"
        t = tokenizer(prompt, truncation = True, max_length = 256)
        return {"input_ids": t["input_ids", "query": prompt]}

    ds = ds.map(tok)
    ds.set_format(type = "torch")
    return ds

ppo_dataset = build_prompt_dataset(ppo_tokenizer)

ppo_config = PPOConfig(
    model_name = SFT_OUTPUT_DIR,
    learning_rate = PPO_LR,
    batch_size = PPO_BATCH_SIZE,
    mini_batch_size = PPO_MINI_BATCH_SIZE,
    cliprange = PPO_CLIP_EPS,
    cliprange_value = PPO_KL_COEF,
    target_kl = 6.0,
    steps = PPO_STEPS,
    log_with = None,
)


ppo_trainer = PPOTrainer(
    config = ppo_config, 
    model = policy_model,
    ref_model = ref_model,
    processing_class = ppo_tokenizer,
    dataset = ppo_dataset,
)

generation_kwargs = {
    "min_length": -1, 
    "top_k": 0.0, 
    "top_p": 1.0,
    "do_sample": True, 
    "pad_token_id": ppo_tokenizer.eos_token_id, 
    "max_new_tokens": 128,
}

for step, batch in enumerate(ppo_trainer.dataloader):
    if step >= PPO_STEPS:
        break

    query_tensors = batch["input_ids"]
    response_tensors = ppo_trainer.generate(query_tensors, **generation_kwargs)
    batch["response"] = [ppo_tokenizer.decode(r.squeeze(), skip_special_tokens = True ) for r in response_tensors]


    texts = [q + r for q, r in zip(batch["query"], batch["response"])]

    with torch.no_grad():
        inputs = ppo_tokenizer(texts, padding = True, truncation = True, return_tensors = "pt")
        scores = reward_model_ppo(**inputs).logits.squeeze(-1)

    rewards = [s for s in scores]

    stats = ppo_trainer.step(list(query_tensors), response_tensors, rewards)
    
    if step % 10 == 0: 
        mean_reward = torch.stack(rewards).mean().item()
        print(f"step {step:4d} | mean_reward = {mean_reward:.3f} "
              f"| kl = {stats.get('objective/kl', float('nan')):.3f} "
              f"| clipfrac = {stats.get('ppo/policy/clipfrac', float('nan')):.3f} ")

ppo_trainer.save_pretrained(PPO_OUTPUT_DIR)
print(f"PPO-tuned policy saved to {PPO_OUTPUT_DIR}")

# Comparison

In [ ]:
# compare the SFT-only model and the PPO-tuned model on a few heldout prompts, with reward model score for each 

EVAL_PROMPTS = [
    "Explain what a black hole is to a 10 year old.",
    "Write a short, polite email declining a meeting invitation.",
    "What are three tips for staying focused while studying?",
    "Summarize the plot of Romeo and Juliet in two sentences.",
]

def generate(model, tokenizer, prompt, max_new_tokens = 128):
    text = f"### Instruction: \n{prompt}\n\n### Repsonse:\n"
    inputs = tokenizer(text, return_tensors = "pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            top_p = 0.9,
            tmeperature = 0.7, 
            pad_token_id = tokenizer.eos_token_id,
        )

    full = tokenizer.decode(output[0], skip_special_tokens = True)
    return full[len(text):].strip()

def score(reward_model, tokenizer, prompt, response):
    text = f"### Instruction: \n{prompt}\n\n## Response:\n{response}"
    inputs = tokenizer(text, return_tensors = "pt", truncation = True )
    with torch.no_grad():
        return reward_model(**inputs).logits.item()

eval_tokenizer = AutoTokenizer.from_pretrained(SFT_OUTPUT_DIR)

if eval_tokenizer.pad_token is None: 
    eval_tokenizer = eval_tokenizer.eos_token


sft_eval_model = AutoModelForCausalLM.from_pretrained(SFT_OUTPUT_DIR).eval()
ppo_eval_model = AutoModelForCausalLM.from_pretrained(PPO_OUTPUT_DIR).eval()
rm_eval_model = AutoModelForSequenceClassification.from_pretrained(RM_OUTPUT_DIR, num_labels = 1).eval()

for prompt in EVAL_PROMPTS:
    print("=" * 80)
    print(f"PROMPT: {prompt}\n")

    sft_response = generate(sft_eval_model, eval_tokenizer, prompt)
    sft_score = score(rm_eval_model, eval_tokenizer, prompt, sft_response)
    print(f"[SFT only]  (reward={sft_score:.3f})\n{sft_response}\n")
    
    ppo_response = generate(ppo_eval_model, eval_tokenizer, prompt)
    ppo_score = score(rm_eval_model, eval_tokenizer, prompt, ppo_response)
    print(f"[PPO-tuned] (reward={ppo_score:.3f})\n{ppo_response}\n")
    
    print(f"Reward delta (PPO - SFT): {ppo_score - sft_score:+.3f}")
    
print("=" * 80)